In [25]:
#Bibliotecas Fase 1
import os
import cv2
import pandas as pd
from hand_landmark_extractor import HandLandmarkExtractor

#Bibliotecas Fase 2
import numpy as np
import time
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, log_loss
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb

RANDOM_SEED = 42

---

## Fase 1 - Preprocessamento e extração de Landmarks

In [26]:
dataset_path = "SignAlphaSet"
output_csv = "asl_landmarks_dataset.csv"

extractor = HandLandmarkExtractor(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.5,
    suppress_warnings=True
)

all_data_frames = []

# Verificar que dataset ainda não existe
if os.path.exists(output_csv):
    print("► Dataset de landmarks já existe!")
else:
    # 1. Iterar sobre todas as pastas do dataset SignAlphaSet (A-Z)
    if not os.path.exists(dataset_path):
        print(f"Erro: A pasta {dataset_path} não foi encontrada.")
        exit()
    
    folders = sorted([f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))])
    
    for label in folders:
        folder_path = os.path.join(dataset_path, label)
        print(f"A processar letra: {label}...")
    
        # Iterar sobre todas as imagens na pasta da letra
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)
            
            # Carrega a imagem
            image = cv2.imread(img_path)
            if image is None:
                continue
    
            # 2. Extrair os landmarks com a biblioteca HandLandmarkExtractor, utilizando lógica do test_camera.py
            hands_data = extractor.process_image_landmarks(image)
    
            if hands_data:
                # 3. Cada landmark contém informação da mão e coordenadas (x, y, z)
                # O método hands_data_to_dataframe já aplana os 21 pontos em 63 colunas + identificação da mão
                df_temp = extractor.hands_data_to_dataframe(hands_data)
                
                # 4. A letra correspondente (label) obtém-se através da pasta
                df_temp['label'] = label
                
                all_data_frames.append(df_temp)
    
    # Resultado Esperado: Um novo dataset estruturado
    if all_data_frames:
        final_df = pd.concat(all_data_frames, ignore_index=True)
        final_df.to_csv(output_csv, index=False)
        print(f"Dataset criado com sucesso: {output_csv}")
        print(f"Total de registos: {len(final_df)}")
    else:
        print("Nenhum landmark foi detetado nas imagens.")
    
    extractor.close()

► Dataset de landmarks já existe!


In [27]:
## Mini App - Aumento de 

In [28]:
def run_data_collector():
    # 1. Configurações
    CSV_OUTPUT = "asl_landmarks_dataset.csv"
    DELAY_AMOSTRA = 0.5 
    extractor = HandLandmarkExtractor(static_image_mode=False, max_num_hands=1)
    cap = cv2.VideoCapture(0)

    # 2. Estado Inicial
    current_label = input("Introduza a letra que vai capturar (ex: A): ").upper()
    recording = False
    last_save_time = 0
    
    print(f"\n--- (Letra {current_label}) ---")
    print("Comandos:")
    print(" [S] - Ligar/Desligar Gravação")
    print(" [N] - Mudar de Letra")
    print(" [Q] - Sair")

    try:
        while cap.isOpened():
            success, frame = cap.read()
            if not success: break

            frame = cv2.flip(frame, 1)
            hands_data = extractor.process_image_landmarks(frame)
            current_time = time.time()

            # Lógica de Gravação Automática
            if recording and hands_data:
                # Verifica se já passaram 0.5 segundos desde a última gravação
                if current_time - last_save_time >= DELAY_AMOSTRA:
                    df = extractor.hands_data_to_dataframe(hands_data)
                    df['label'] = current_label
                    
                    file_exists = os.path.exists(CSV_OUTPUT)
                    df.to_csv(CSV_OUTPUT, mode='a', header=not file_exists, index=False)
                    
                    last_save_time = current_time
                    print(f"Amostra guardada para '{current_label}'...")

            # Feedback Visual
            if hands_data:
                frame = extractor.draw_landmarks(frame, hands_data)
            
            # HUD
            status_color = (0, 0, 255) if recording else (0, 255, 0)
            status_text = "GRAVANDO (0.5s)" if recording else "PARADO"
            
            cv2.putText(frame, f"Status: {status_text}", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
            cv2.putText(frame, f"Letra: {current_label}", (10, 60), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            
            cv2.imshow("Coletor de Dados Automático", frame)
            key = cv2.waitKey(1) & 0xFF

            # Teclas de controlo
            if key == ord('s'):
                recording = not recording
                print(f"Estado da gravação: {'LIGADO' if recording else 'DESLIGADO'}")
            
            elif key == ord('n'):
                recording = False # Para a gravação por segurança ao mudar
                current_label = input("\nIntroduza a nova letra: ").upper()
                print(f"Letra alterada para: {current_label}")
            
            elif key == ord('q'):
                break

    finally:
        cap.release()
        cv2.destroyAllWindows()
        extractor.close()

#run_data_collector()

---

## Fase 2 - Desenvolvimento e Treino do Modelo de Classificação

In [29]:
def feature_engineering(df):
    df = df.copy()
    
    # 1. Remover J e Z
    df = df.query("label not in ['J', 'Z']").reset_index(drop=True)

    #2. Extração das colunas por eixo
    x_cols = [c for c in df.columns if c.endswith("_x")]
    y_cols = [c for c in df.columns if c.endswith("_y")]
    z_cols = [c for c in df.columns if c.endswith("_z")]
    lm_cols = x_cols + y_cols + z_cols
    
    # 3. Espelhamento da mão esquerda e criar feature binária (0 e 1 para cada tipo de mão)
    if "hand" in df.columns:
        mao_esquerda = df["hand"].str.lower() == "left"
        df.loc[mao_esquerda, x_cols] = 1.0 - df.loc[mao_esquerda, x_cols]
        df["is_right"] = df["hand"].str.lower().map({"right": 1, "left": 0})
        df = df.drop(columns=["hand"])
    
    # 3. Normalização: Centrar no WRIST e escalar pela distância WRIST -> MIDDLE_FINGER_MCP
    df[x_cols] = df[x_cols].sub(df["WRIST_x"], axis=0)
    df[y_cols] = df[y_cols].sub(df["WRIST_y"], axis=0)
    df[z_cols] = df[z_cols].sub(df["WRIST_z"], axis=0)
    
    scale = np.sqrt(df["MIDDLE_FINGER_MCP_x"]**2 + df["MIDDLE_FINGER_MCP_y"]**2 + df["MIDDLE_FINGER_MCP_z"]**2)
    df[lm_cols] = df[lm_cols].div(scale.replace(0, 1), axis=0)
    
    return df

##### 1 º - Filtragem de Letras de Movimento
Removemos a letras 'J' e 'Z' pois o modelo utiliza letras estáticas para previsao e estas envolvem movimento para serem identificadas e assim reduzimos o dominio das features e as chances de o modelo enganar-se.


##### 2º - Agrupamento por Eixos
Fazemos o parsing e agrupamento das colunas em listas específicas de eixos ($x, y, z$). Isto permite a aplicação de transformações matemáticas em massa apenas nas coordenadas espaciais. Isto garante a eficiência dos cálculos e protege a integridade dos restantes dados($i.e$ label da letra e lateralidade da mão) que devem permanecer inalterados.

##### 3º Unificação e Invariância Geométrica
- **Espelhamento**: Invertemos o eixo X nas mãos esquerdas para que o modelo aprenda todos os gestos como sendo de uma mão direita, simplificando o treino.
  
- **Centralização**: Subtraimos as coordenadas do pulso (WRIST) a todos os pontos, fixando a mão na origem $(0,0,0)$ e tornando o modelo invariante à posição no ecrã. Como grande parte das fotos do dataset são fotos tiradas no centro do imagem/frame, se nao tivemos esta questão de centralização dos dados considerando apresentando a imagem alem do centro do frame nao iria detetar corretamente as letras.

$$P'_i = P_i - P_{wrist}$$ 
  
- **Escalonamento**: Normalizamos, ou seja, redimensionamos proporcionalmente todos os pontos da mão com base numa medida de referência constante (a distância entre o pulso e a base do dedo médio). Para a câmara, uma mão a perto que esteja perto parece enorme, enquanto uma mão mais longe minúscula. Sem normalização, os valores das coordenadas seriam drasticamente diferentes, fazendo com que o modelo realiza-se previsões "erradas".
O escalonamento é alcancado da seguinte forma:

<center><img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTx1F7LeZlUokvNtB-Ip9yX4HO03GWOkqEEtA&s"></center>

  **Scale ($S$)** | Começamos por calcular o comprimento do vetor que vai da origem (o pulso, já centrado em $0,0,0$) até à base do dedo médio (Medial Canthal Point - MCP).
      


$$S = \sqrt{x_{mcp}^2 + y_{mcp}^2 + z_{mcp}^2}$$
  **df[lm_cols]** | Para cada ponto da mão $P_{i}$ presente na lista lm_cols, aplicamos a divisão pelo fator de escala para obter a nova coordenada normalizada $P'_i$. Isto garante que a distância de referência passe a ser sempre igual a 1, independentemente da distância à câmara.
  
$$x'_i = \frac{x_i}{S}, \quad y'_i = \frac{y_i}{S}, \quad z'_i = \frac{z_i}{S}$$



In [30]:
df_original = pd.read_csv(output_csv)
df = feature_engineering(df_original)
print("✓ Feature Engineering aplicado com sucesso")

✓ Feature Engineering aplicado com sucesso


In [31]:
X = df.drop(columns=["label"])
y = df["label"]

In [32]:
x_cols = [c for c in X.columns if c.endswith("_x")]
y_cols = [c for c in X.columns if c.endswith("_y")]
z_cols = [c for c in X.columns if c.endswith("_z")]

coords_cols = x_cols + y_cols + z_cols

restantes = [c for c in X.columns if c not in coords_cols]

print(f"Contagem de colunas por eixo:")
print(f"  - Eixo X: {len(x_cols)} colunas")
print(f"  - Eixo Y: {len(y_cols)} colunas")
print(f"  - Eixo Z: {len(z_cols)} colunas")
print(f"Total de coordenadas (X + Y + Z): {len(coords_cols)} valores")
print(f"\nColunas restantes: {len(restantes)} {restantes}")
print(f"Dimensão total de X: {X.shape[1]} colunas")

Contagem de colunas por eixo:
  - Eixo X: 21 colunas
  - Eixo Y: 21 colunas
  - Eixo Z: 21 colunas
Total de coordenadas (X + Y + Z): 63 valores

Colunas restantes: 1 ['is_right']
Dimensão total de X: 64 colunas


---
$X$ refere-se às features relevantes a serem utilizadas pelo modelo para realizar as previsões das letras. Neste caso, as coordenadas em $x,y$ e $z$ e lateralidade são as features.

### Enconding das labels (Target)
Uma vez que os algoritmos de $ML$ não conseguem processar texto diretamente, é necessário converter as labels em formatos numéricos. Este processo é realizado em três etapas:

- **Criação de Índices**: Identificamos todas as classes únicas no dataset e ordenamo-las alfabeticamente para garantir consistência.

- **Mapeamento Bidirecional**: Criamos dois dicionários **(label_map** e **inv_label_map)**. O primeiro permite converter as letras em números para o treino, enquanto o segundo permite traduzir as previsões numéricas do modelo de volta para letras compreensíveis por pessoas.

- **Transformação**: A coluna "$label$" original é mapeada para o array **y_encoded**, resultando num vetor de inteiros onde cada número representa uma classe específica.

In [33]:
letras = sorted(df['label'].unique())

label_map = {label: i for i, label in enumerate(letras)}
inv_label_map = {i: label for label, i in label_map.items()}

y_encoded = df['label'].map(label_map).values

In [43]:
header_len = 60
print("=" * header_len)
print("Label Enconding".center(header_len))
print("=" * header_len)

print(f"Total de Letras: {len(letras)}")
print(f"Letras detedas: {', '.join(letras)}")
print("-" * header_len)

print("Dicionários:")
print(f"{'LETRA':<7} ➔ {'ÍNDICE':<10} | {'ÍNDICE':<5} ➔ {'LETRA':<10}")
print("-" * 45)
for (letra, idx) in label_map.items():
    print(f"  {letra:<5} ➔ {idx:<10} |   {idx:<5} ➔ {letra:<10}")

print("-" * header_len)

print("Vetor alvo (y_encoded):")
print(f"valores Únicos:  {np.unique(y_encoded)}")
print(f"formato (Shape): {y_encoded.shape[0]} amostras totais")
print("=" * header_len)

                      Label Enconding                       
Total de Letras: 24
Letras detedas: A, B, C, D, E, F, G, H, I, K, L, M, N, O, P, Q, R, S, T, U, V, W, X, Y
------------------------------------------------------------
Dicionários:
LETRA   ➔ ÍNDICE     | ÍNDICE ➔ LETRA     
---------------------------------------------
  A     ➔ 0          |   0     ➔ A         
  B     ➔ 1          |   1     ➔ B         
  C     ➔ 2          |   2     ➔ C         
  D     ➔ 3          |   3     ➔ D         
  E     ➔ 4          |   4     ➔ E         
  F     ➔ 5          |   5     ➔ F         
  G     ➔ 6          |   6     ➔ G         
  H     ➔ 7          |   7     ➔ H         
  I     ➔ 8          |   8     ➔ I         
  K     ➔ 9          |   9     ➔ K         
  L     ➔ 10         |   10    ➔ L         
  M     ➔ 11         |   11    ➔ M         
  N     ➔ 12         |   12    ➔ N         
  O     ➔ 13         |   13    ➔ O         
  P     ➔ 14         |   14    ➔ P         
  Q     ➔

---

### Divisão de Treino, Teste e Validação

Vamos fazer a divisão das amostras para treino, teste e validação, utilizando o método da biblioteca **scikit-learn**, obter um divisão de **70% Treino**, **15% Teste**  e **15% Validação**. Como a biblioteca não disponibiliza uma fazer logo a divisão, começamos por fazer uma divisão e dividimos novamente.

In [55]:
PERCENT_TEST = 0.15

In [56]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y_encoded, test_size=PERCENT_TEST, random_state=RANDOM_SEED, stratify=y_encoded)

No primeiro split, reservamos logo **15%** para o teste final (X_test). Sobram **85% dos dados originais para serem divididos entre treino e validação**. Para garatir que o conjunto de validação tenha o mesma percentagem que teste, podemos utilizar a seguinte fórmula:

$$\frac{PERCENT TEST(= 15\%)}{100\%  - PERCENT TEST (= 85\%)} \approx 0.176$$

In [57]:
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.176, random_state=RANDOM_SEED, stratify=y_train_val)

In [46]:
def print_split_summary(X_train, X_val, X_test, total_size):
    header = "Resumo da divisão do Dataset"
    print("=" * 50)
    print(header.center(50))
    print("=" * 50)
    
    sets = [
        ("Treino", X_train, "70%"),
        ("Validação", X_val, "15%"),
        ("Teste", X_test, "15%")
    ]
    
    for name, data, target in sets:
        size = len(data)
        perc = (size / total_size) * 100
        print(f"{name:<12}: {size:<6} amostras | {perc:>5.1f}% (Alvo: {target})")
    
    print("-" * 50)
    print(f"TOTAL GERAL  : {total_size} amostras")
    print("=" * 50)

# Executar o print
print_split_summary(X_train, X_val, X_test, len(X))

           Resumo da divisão do Dataset           
Treino      : 16728  amostras |  70.0% (Alvo: 70%)
Validação   : 3574   amostras |  15.0% (Alvo: 15%)
Teste       : 3583   amostras |  15.0% (Alvo: 15%)
--------------------------------------------------
TOTAL GERAL  : 23885 amostras


In [10]:
models_config = {
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {"n_estimators": [100, 200], "max_depth": [None, 10]}
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {"n_neighbors": [3, 5, 7], "weights": ["distance"]}
    },
    "XGBoost": {
        "model": xgb.XGBClassifier(random_state=42, eval_metric='mlogloss'),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [3, 6],
            "learning_rate": [0.1, 0.2]
        }
    }
}

best_model = None
best_f1 = -1

print("A INICIAR GRIDSEARCHCV")
for name, cfg in models_config.items():
    grid = GridSearchCV(cfg["model"], cfg["params"], cv=3, n_jobs=-1, scoring='f1_weighted', verbose=1)
    grid.fit(X_train, y_train)
    
    val_preds = grid.predict(X_val)
    val_f1 = f1_score(y_val, val_preds, average='weighted')
    print(f"-> {name:15} | Val F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_model = grid.best_estimator_
        best_name = name

print(f"\n--- MELHOR MODELO: {best_name} ---")

# 1. Cross-Validation no set de treino Total
cv_scores = cross_val_score(best_model, X_train_val, y_train_val, cv=5)
print(f"Cross-Validation Accuracy (5 folds): {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# 2. Avaliação Final no Conjunto de Teste (Dados nunca vistos)
test_preds = best_model.predict(X_test)
acc = accuracy_score(y_test, test_preds)

print(f"\nAccuracy Final no Teste: {acc:.4%}")
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, test_preds, target_names=classes))

# 3. Visualização: Matriz de Confusão
fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test, 
    display_labels=classes, 
    cmap='Blues', 
    ax=ax
)
plt.title(f"Matriz de Confusão - {best_name}")
plt.show()

# Exportação do Modelo
export_data = {'model': best_model, 'inv_label_map': inv_label_map}
with open('melhor_modelo.pkl', 'wb') as f:
    pickle.dump(export_data, f)

SyntaxError: invalid non-printable character U+00A0 (2878168457.py, line 59)

## Accuracy (Acurácia)

<center>
<span style="font-size: 24px;">
$\text{Accuracy} = \frac{\text{nº de previsões corretas}}{\text{nº total de previsões}}$
</span>
</center>


A métrica principal utilizada foi a acurácia, uma vez que o problema consiste numa classificação multiclasse com classes equilibradas, onde todas as letras têm igual importância.

    models_config = {
        'SVM': {
            'model': SVC(probability=True, random_state=42),
            'params': {
                'C': [1, 10],
                'kernel': ['rbf', 'linear']
            }
        },
        'XGBoost': {
            'model': xgb.XGBClassifier(random_state=42, eval_metric='mlogloss'),
            'params': {
                'n_estimators': [100, 200],
                'max_depth': [3, 6],
                'learning_rate': [0.1, 0.2]
            }
        },
        'NeuralNetwork': {
            'model': MLPClassifier(random_state=42, max_iter=500),
            'params': {
                'hidden_layer_sizes': [(128, 64), (64, 64)],
                'alpha': [0.0001, 0.001]
            }
        },
        'RandomForest': {
            'model': RandomForestClassifier(random_state=42),
            'params': {
                'n_estimators': [100, 200],
                'max_depth': [None, 20]
            }
        }
    }



          "KNN": {
            "model": KNeighborsClassifier(),
            "params": {
                "n_neighbors": [3, 5, 7, 9, 11],
                "weights": ["uniform", "distance"],
                "metric": ["euclidean", "manhattan", "minkowski"],
                "leaf_size": [20, 30, 40]
            }
        },
        "XGBoost": {
            "model": xgb.XGBClassifier(
                random_state=42, 
                eval_metric="mlogloss",
                tree_method="hist",
                n_jobs=-1 # Mais rápido para datasets grandes
            ),
            "params": {
                "n_estimators": [100, 300],
                "max_depth": [3, 6],
                "learning_rate": [0.1, 0.2],
                "subsample": [0.8, 1.0],
                "colsample_bytree": [0.8, 1.0],
                #"gamma": [0, 0.1, 0.2],  # Regularização para evitar overfitting
                #"min_child_weight": [1, 3, 5]
            }
        },
        "SVM": {
            "model": SVC(probability=True, random_state=42),
            "params": {
                "C": [0.1, 1, 10, 100],
                "kernel": ["rbf", "poly", "linear"],
                "gamma": ["scale", "auto", 0.01, 0.001],
                "degree": [3, 5]  # Apenas relevante se o kernel for 'poly'
            }
        }
    }